In [ ]:
# Coordinate enrichment for "Want to go.csv"
#
# Adds Latitude/Longitude columns using OpenStreetMap's free Nominatim
# geocoder (no API key). Run cells top to bottom. The full run takes
# roughly 7-10 minutes for ~400 rows (Nominatim allows 1 request/second),
# but progress is saved as it goes, so it is safe to stop and re-run.

# Install dependencies into THIS notebook's kernel (fixes ModuleNotFoundError).
%pip install -q pandas geopy requests

In [ ]:
import time
from pathlib import Path

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# The real CSV lives in Takeout/Saved/ next to this notebook.
INPUT_FILE = Path("Takeout/Saved/Want to go.csv")
OUTPUT_FILE = Path("Takeout/Saved/Want to go_geocoded.csv")

assert INPUT_FILE.exists(), (
    f"Could not find {INPUT_FILE.resolve()} - "
    "run this notebook from the travel/ folder."
)

geolocator = Nominatim(user_agent="want_to_go_trip_planner")
print(f"Input: {INPUT_FILE.resolve()}")

In [ ]:
# Load the CSV (resume from a previous partial run if the output exists).
if OUTPUT_FILE.exists():
    df = pd.read_csv(OUTPUT_FILE)
    print(f"Resuming from existing {OUTPUT_FILE.name}")
else:
    df = pd.read_csv(INPUT_FILE)
    df["Latitude"] = pd.NA
    df["Longitude"] = pd.NA
    df["GeocodeNote"] = ""

# Drop fully blank rows (the CSV has some).
df = df.dropna(subset=["Title"]).reset_index(drop=True)

already_done = df["Latitude"].notna().sum()
print(f"{len(df)} places total, {already_done} already geocoded")
df.head()

In [ ]:
def geocode_place(query, retries=3):
    for attempt in range(retries):
        try:
            return geolocator.geocode(query, timeout=10)
        except (GeocoderTimedOut, GeocoderServiceError):
            time.sleep(2)
    return None


# Main geocoding loop. Safe to interrupt: progress saves every 10 rows,
# and re-running skips rows that already have coordinates.
total = len(df)
for i, row in df.iterrows():
    if pd.notna(row.get("Latitude")):
        continue  # already geocoded

    title = str(row["Title"]).strip()
    print(f"[{i + 1}/{total}] Geocoding: {title}")

    location = geocode_place(title)

    if location:
        df.at[i, "Latitude"] = location.latitude
        df.at[i, "Longitude"] = location.longitude
        df.at[i, "GeocodeNote"] = ""
    else:
        df.at[i, "GeocodeNote"] = "NOT FOUND - check manually"

    if i % 10 == 0:
        df.to_csv(OUTPUT_FILE, index=False)

    time.sleep(1)  # respect Nominatim's 1 request/second limit

df.to_csv(OUTPUT_FILE, index=False)
print(f"\nDone! Saved to {OUTPUT_FILE}")

In [ ]:
# Second pass: resolve rows Nominatim couldn't find, using the Google Maps
# CID embedded in each place's URL instead of the place name.
#
# Each Google Maps link contains ...!1s0x<hex>:0x<hex>. The part after the
# colon is Google's internal ID (CID) for that exact place. The full Maps
# page no longer puts @lat,lng in the redirect URL, so we hit the small
# embed endpoint instead and parse [lat,lng] from the HTML body.
# No API key; exact place match. Updates the geocoded CSV in place.

import re

import requests

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}

CID_RE = re.compile(r"!1s0x[0-9a-fA-F]+:0x([0-9a-fA-F]+)")
# Embed HTML contains the place coordinates as a [lat,lng] array.
COORD_RE = re.compile(r"\[(-?\d+\.\d+),(-?\d+\.\d+)\]")


def _valid_coords(lat: float, lng: float) -> bool:
    if not (-90.0 <= lat <= 90.0 and -180.0 <= lng <= 180.0):
        return False
    # Reject null island / empty placeholders.
    return not (lat == 0.0 and lng == 0.0)


def resolve_via_cid(url, retries=3):
    match = CID_RE.search(str(url))
    if not match:
        return None, None

    cid_dec = int(match.group(1), 16)
    lookup_url = f"https://maps.google.com/maps?cid={cid_dec}&output=embed&hl=en"

    for attempt in range(retries):
        try:
            resp = requests.get(lookup_url, headers=HEADERS, timeout=15)
            coord_match = COORD_RE.search(resp.text)
            if not coord_match:
                return None, None
            lat, lng = float(coord_match.group(1)), float(coord_match.group(2))
            if _valid_coords(lat, lng):
                return lat, lng
            return None, None
        except requests.RequestException:
            time.sleep(2)
    return None, None


# Reload the first pass's output and retry only the NOT FOUND rows.
df = pd.read_csv(OUTPUT_FILE)
to_fix = df[df["GeocodeNote"].fillna("").str.contains("NOT FOUND")]
print(f"Attempting to resolve {len(to_fix)} rows via Google CID embed...")

# Spot-check a few known places before the full run.
spot_titles = [
    "Great Sphinx of Giza",
    "Historic Sanctuary of Machu Picchu",
    "Memorial and Museum Auschwitz-Birkenau",
]
for title in spot_titles:
    spot = to_fix[to_fix["Title"] == title]
    if spot.empty:
        print(f"  [spot] {title}: not in NOT FOUND set (already resolved?)")
        continue
    lat, lng = resolve_via_cid(spot.iloc[0]["URL"])
    print(f"  [spot] {title} -> {lat}, {lng}")
    time.sleep(1)

resolved = 0
for i, row in to_fix.iterrows():
    title = row["Title"]
    lat, lng = resolve_via_cid(row["URL"])

    if lat is not None:
        df.at[i, "Latitude"] = lat
        df.at[i, "Longitude"] = lng
        df.at[i, "GeocodeNote"] = "resolved via Google CID"
        resolved += 1
        print(f"  [OK] {title} -> {lat}, {lng}")
    else:
        print(f"  [--] {title} still unresolved")

    time.sleep(1)  # be polite

df.to_csv(OUTPUT_FILE, index=False)
print(f"\nResolved {resolved}/{len(to_fix)} additional rows.")
print(f"Saved back to {OUTPUT_FILE}")


In [ ]:
# Summary: how many geocoded, and which ones need manual attention.
geocoded = df["Latitude"].notna().sum()
missing = df[df["GeocodeNote"].fillna("").str.contains("NOT FOUND")]
print(f"Geocoded: {geocoded}/{len(df)}")
if len(missing):
    print(f"\n{len(missing)} places couldn't be geocoded automatically:")
    print(missing["Title"].to_string(index=False))